# AutoMLChain - Local Provider Test

Este notebook testa o **LocalProvider** do AutoMLChain - fine-tuning local usando sua GPU.

### Requisitos:
- GPU NVIDIA com CUDA OU Mac Silicon (Apple M1/M2)
- ~5GB de espaço em disco

### Tempo estimado:
- Instalação dependências: ~10 minutos
- Download modelo: ~5 minutos
- Fine-tuning (1 epoch, 10 samples): ~15-30 minutos (depende da GPU)

## 1. Setup e Instalação

In [ ]:
# Setup - Instalar dependências do LocalProvider
!pip install torch transformers peft accelerate datasets bitsandbytes huggingface_hub sentencepiece --quiet
!pip install "torchao>=0.16.0" --quiet

import os
import sys

# Adicionar path do repositório clonado
sys.path.insert(0, '/content/automlchain')

# Alternativa: instalar do git se não tiver o código
try:
    from automlchain import AutoMLPipeline
except ModuleNotFoundError:
    print("Instalando automlchain do git...")
    !pip install git+https://github.com/gumeeee/automlchain.git --quiet
    from automlchain import AutoMLPipeline

print("✅ AutoMLChain instalado!")

# Verificar GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Preparar Dataset de Teste

In [ ]:
import json

# Dataset pequeno para teste rápido
# Usando instruction format para modelo aprender a tarefa
test_data = [
    {"input": "I love this product, it's amazing!", "output": "positive"},
    {"input": "This is the worst thing I ever bought.", "output": "negative"},
    {"input": "Really enjoyed using this, highly recommend.", "output": "positive"},
    {"input": "Not bad, but could be better.", "output": "neutral"},
    {"input": "Terrible experience, never again.", "output": "negative"},
    {"input": "It's okay, nothing special.", "output": "neutral"},
    {"input": "Best purchase I ever made!", "output": "positive"},
    {"input": "Very disappointed with the quality.", "output": "negative"},
    {"input": "Works as expected, good value.", "output": "positive"},
    {"input": "Would not recommend to anyone.", "output": "negative"},
]

# Salvar dataset
os.makedirs("data", exist_ok=True)
with open("data/test_sentiment.jsonl", "w") as f:
    for item in test_data:
        f.write(json.dumps(item) + "\n")

print(f"✅ Dataset salvo: {len(test_data)} samples")

## 3. Criar Pipeline com LocalProvider

In [ ]:
# Criar pipeline com LocalProvider
pipeline = AutoMLPipeline(provider="local")

print("✅ LocalProvider pronto!")
print(f"Modelo padrão: {pipeline.provider.model}")
print(f"Output dir: {pipeline.provider.output_dir}")

## 4. Iniciar Fine-tuning

In [ ]:
print("🚀 Iniciando fine-tuning local...")
print(f"Modelo: TinyLlama/TinyLlama_v1.1")
print(f"Dataset: data/test_sentiment.jsonl")
print("-" * 50)

# Iniciar treinamento - T4 otimizado
job = pipeline.train(
    model="TinyLlama/TinyLlama_v1.1",
    training_file="data/test_sentiment.jsonl",
    hyperparameters={
        "epochs": 3,
        "batch_size": 1,           # Reduzido para T4 (~16GB VRAM)
        "max_seq_length": 128,     # Reduzido para T4
        "gradient_accumulation_steps": 8,  # Compensa batch menor
        "learning_rate": 1e-4,
        "lora_rank": 8,            # Reduzido para menor uso de memória
    }
)

print(f"
✅ Job iniciado!")
print(f"   Job ID: {job.job_id}")
print(f"   Modelo: {job.model}")

## 5. Monitorar Progresso

**Nota**: Esta célula fica em loop. Execute e aguarde. Pressione Kernel > Interrupt para cancelar.

In [ ]:
import time

print("📊 Monitorando progresso...")
print("Pressione Kernel > Interrupt para cancelar")
print()

while True:
    status = pipeline.get_status(job.job_id)
    
    print(f"[{time.strftime('%H:%M:%S')}] Status: {status.status} | Progress: {status.progress:.1f}%")
    
    if status.status == "completed":
        print("\n🎉 Treinamento concluído!")
        break
    elif status.status == "failed":
        print(f"\n❌ Falhou: {status.error}")
        break
    elif status.status == "cancelled":
        print("\n⚠️ Treinamento cancelado pelo usuário")
        break
    
    time.sleep(30)  # Verificar a cada 30 segundos

## 6. Testar Inferência (após conclusão)

In [ ]:
# Testar o modelo fine-tuned
print("🧪 Testando inferência com modelo fine-tuned...")
print()

# Usar instruction format para melhor resultados
test_prompts = [
    "### Input:\nI really love this product!\n### Output:\n",
    "### Input:\nThis is awful, I hate it.\n### Output:\n",
    "### Input:\nIt's okay, nothing special.\n### Output:\n"
]

for prompt in test_prompts:
    print(f"📝 Input: {prompt.strip()}")
    try:
        result = pipeline.provider.infer(
            prompt=prompt,
            job_id=job.job_id,
            max_tokens=20
        )
        print(f"   Output: {result}")
    except Exception as e:
        print(f"   ⚠️ Erro: {e}")
    print()

## 7. Verificar Outputs

In [ ]:
# Verificar outputs salvos
import os

output_base = "./outputs/checkpoints"

if os.path.exists(output_base):
    print(f"📁 Outputs salvos em: {output_base}")
    print()
    
    for job_dir in os.listdir(output_base):
        job_path = os.path.join(output_base, job_dir)
        if os.path.isdir(job_path):
            print(f"Job: {job_dir}")
            for f in os.listdir(job_path):
                print(f"   - {f}")
            print()
else:
    print("Outputs ainda não disponíveis...")

---

## 📝 Notas

### ⚠️ Limitações (Proof of Concept)

Este é um **proof of concept**. Para uso em produção:

| Aspecto | Atual | Para Produção |
|---------|-------|--------------|
| Dataset | 10 samples | 1000+ samples |
| Epochs | 3 | 5-10 |
| Avaliação | Manual | Automatizada |
| Modelo | TinyLlama | Llama-3.2-1B ou maior |

### Modelos Recomendados

| Modelo | Tamanho | VRAM | Uso |
|--------|---------|------|-----|
| TinyLlama/TinyLlama_v1.1 | 1GB | 2GB | Testes rápidos |
| Qwen/Qwen2-0.5B | 1GB | 2GB | Bom custo-benefício |
| meta-llama/Llama-3.2-1B | 1GB | 2GB | Mais capaz |

### Troubleshooting

**"CUDA out of memory"**:
```python
hyperparameters = {
    "batch_size": 1,  # Reduzir
    "max_seq_length": 128,  # Reduzir
}
```

**Sem GPU**:
O fine-tuning em CPU é muito lento. Recomendamos usar GPU do Colab (Runtime > Change runtime type > GPU).